# Lab 1: Deploy and Benchmark Your First Model

In this lab, you will:
1. **Deploy** Meta Llama 3.1 8B Instruct from SageMaker JumpStart
2. **Configure** a workload specification that defines your traffic pattern
3. **Run** an automated benchmark job using NVIDIA AIPerf
4. **Analyze** the results - TTFT, ITL, throughput, and latency percentiles

By the end of this lab, you'll understand the complete benchmarking workflow and have a baseline set of metrics to compare against in later labs.

---

## Why Automated Benchmarking?

Manual load testing is error-prone and time-consuming. SageMaker AI's Automated Benchmarking:
- Uses **NVIDIA AIPerf** - an industry-standard benchmarking tool - under the hood
- Produces **statistically rigorous** results with multi-run confidence intervals
- Measures **LLM-specific metrics** (TTFT, ITL) that generic load testers miss
- Runs in a **managed environment** - no infrastructure to set up or tear down

## Setup

Install dependencies and initialize the SageMaker environment.

In [ ]:
%pip install -r ../requirements.txt --quiet --no-warn-conflicts

In [ ]:
import json
import time
import os
import boto3
import pandas as pd
from datetime import datetime

# Add parent directory to path for utils
import sys
sys.path.append("..")
from utils import (
    get_execution_role,
    wait_for_endpoint,
    wait_for_benchmark_job,
    parse_benchmark_results,
    extract_metrics,
    format_metrics_table,
)

role = get_execution_role()
region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account_id}"  # SageMaker default bucket convention
timestamp = datetime.now().strftime("%m%d%H%M%S")

# Initialize clients
sm_client = boto3.client("sagemaker", region_name=region)
smr_client = boto3.client("sagemaker-runtime", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"Region:          {region}")
print(f"Role:            {role}")
print(f"Default bucket:  {bucket}")
print(f"Timestamp:       {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 1. Deploy Meta Llama 3.1 8B Instruct

The shared endpoint was created in **Lab 0**. Here we deploy the model by creating an
**Inference Component** — this loads the model weights onto the GPU and starts serving.

**Model choice rationale:**
- **Meta Llama 3.1 8B Instruct** — widely-used instruction-tuned model
- bf16 weights (~16GB) fit comfortably on the 4x A10G GPUs (96GB total)
- Supports the OpenAI-compatible chat completions API (required for benchmarking)


In [ ]:
# ---------------------------------------------------------------
# Load shared endpoint from Lab 0
# ---------------------------------------------------------------
with open("../results/endpoint_info.json", "r") as f:
    endpoint_info = json.load(f)

endpoint_name = endpoint_info["endpoint_name"]
print(f"Using shared endpoint: {endpoint_name}")

# ---------------------------------------------------------------
# Model Configuration
# ---------------------------------------------------------------
# LMI Deep Learning Container (DLC) with vLLM serving engine
lmi_image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124"

# HuggingFace model to serve
hf_model_id = "meta-llama/Llama-3.1-8B-Instruct"

# HuggingFace token — required for gated models like Llama
hf_token = os.environ.get("HF_TOKEN", "")

# Unique IC name for this lab
inference_component_name = f"{endpoint_name}-lab1-llama31-8b"

print(f"Model:     {hf_model_id}")
print(f"Container: {lmi_image_uri.split('/')[1]}")
print(f"IC name:   {inference_component_name}")


In [ ]:
# ---------------------------------------------------------------
# Deploy model as an Inference Component on the shared endpoint
# ---------------------------------------------------------------
sm_client.create_inference_component(
    InferenceComponentName=inference_component_name,
    EndpointName=endpoint_name,
    VariantName="AllTraffic",
    Specification={
        "Container": {
            "Image": lmi_image_uri,
            "Environment": {
                "HF_MODEL_ID": hf_model_id,
                "HF_TOKEN": hf_token,
                "OPTION_ROLLING_BATCH": "vllm",
                "OPTION_MAX_ROLLING_BATCH_SIZE": "32",
                "OPTION_TENSOR_PARALLEL_DEGREE": "max",
                "OPTION_MAX_MODEL_LEN": "8192",
                "OPTION_DTYPE": "bf16",
            },
        },
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": 4,
            "MinMemoryRequiredInMb": 1024,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)
print(f"⏳ Deploying model onto shared endpoint (~5-8 min)...")
print(f"   IC: {inference_component_name}")
print(f"   Endpoint: {endpoint_name}")

# Wait for IC to be InService
import time
while True:
    response = sm_client.describe_inference_component(
        InferenceComponentName=inference_component_name
    )
    status = response["InferenceComponentStatus"]
    print(f"  IC Status: {status}")
    if status == "InService":
        print(f"\n✅ Model deployed! IC '{inference_component_name}' is InService.")
        break
    elif status == "Failed":
        reason = response.get("FailureReason", "Unknown")
        raise RuntimeError(f"❌ IC creation failed: {reason}")
    time.sleep(30)


### Verify the endpoint with a test inference

Before benchmarking, let's confirm the endpoint is responding correctly using the OpenAI-compatible chat completions format. This is the same API format that the benchmarking service will use.

In [ ]:
# Test inference using OpenAI-compatible chat completions format
payload = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Amazon SageMaker AI? Answer in 2-3 sentences."}
    ],
    "max_tokens": 150,
    "temperature": 0.7,
    "stream": False,
}

response = smr_client.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=inference_component_name,
    ContentType="application/json",
    Body=json.dumps(payload),
)

result = json.loads(response["Body"].read().decode("utf-8"))
print("Model response:")
print("-" * 40)
print(result["choices"][0]["message"]["content"])
print("-" * 40)
print(f"\nUsage: {result.get('usage', {})}")


## 2. Create a Workload Configuration

A **workload configuration** defines the traffic pattern that the benchmark will simulate. Think of it as describing your expected production workload:

- **Token distributions**: How many input/output tokens per request?
- **Concurrency**: How many simultaneous users?
- **Streaming**: Are responses streamed token-by-token?
- **Tokenizer**: Which tokenizer to use for accurate token counting?

The benchmarking service uses this configuration with NVIDIA AIPerf to generate synthetic requests that match your described traffic pattern.

> 💡 **Tip**: Match these parameters to your actual production traffic for meaningful results. In later labs, we'll explore how varying these parameters changes performance characteristics.

### Set your HuggingFace token

Used to load the model container.


In [ ]:
# Create the workload configuration
workload_config_name = f"bench-config-baseline-{timestamp}"

# This workload spec represents a typical chatbot workload:
# - ~550 input tokens (a paragraph-length question with context)
# - ~150 output tokens (a concise answer)
# - 10 concurrent users
# - Streaming enabled (most chat UIs stream responses)
workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "prompt_input_tokens_mean": 550,
        "prompt_input_tokens_stddev": 150,
        "output_tokens_mean": 150,
        "output_tokens_stddev": 50,
        "concurrency": 10,
        "request_count": 100,
        "streaming": True,
    },
    "tooling": {"api_standard": "openai"},
}

response = sm_client.create_ai_workload_config(
    AIWorkloadConfigName=workload_config_name,
    AIWorkloadConfigs={
        "WorkloadSpec": {"Inline": json.dumps(workload_spec)}
    },
)

workload_config_arn = response["AIWorkloadConfigArn"]
print(f"Workload config created: {workload_config_name}")
print(f"   ARN: {workload_config_arn}")
print(f"\nWorkload parameters:")
print(f"   Input tokens:  550 +/- 150 (mean +/- stddev)")
print(f"   Output tokens: 150 +/- 50")
print(f"   Concurrency:   10 simultaneous requests")
print(f"   Request count: 100 total requests")
print(f"   Streaming:     True")
print(f"   Tokenizer:     meta-llama/Llama-3.1-8B-Instruct")

## 3. Run the Benchmark Job

Now we run the actual benchmark. The `CreateAIBenchmarkJob` API:
1. Spins up a benchmarking container in your account
2. Sends synthetic requests to your endpoint using NVIDIA AIPerf
3. Collects detailed metrics (TTFT, ITL, latency percentiles, throughput)
4. Writes results to your S3 bucket
5. Cleans up the benchmarking infrastructure

The job typically completes in **3-8 minutes** depending on the workload configuration.

> 💡 **Tip**: The benchmark job measures your endpoint's performance without modifying it. Your endpoint remains available for normal traffic during benchmarking.

In [ ]:
# Define S3 output location for benchmark results
s3_output_location = f"s3://{bucket}/benchmark-results/lab1-baseline-{timestamp}/"

# Create the benchmark job
benchmark_job_name = f"bench-baseline-{timestamp}"

response = sm_client.create_ai_benchmark_job(
    AIBenchmarkJobName=benchmark_job_name,
    BenchmarkTarget={
        "Endpoint": {
            "Identifier": endpoint_name,
            "InferenceComponents": [
                {"Identifier": inference_component_name}
            ]
        }
    },
    OutputConfig={
        "S3OutputLocation": s3_output_location
    },
    AIWorkloadConfigIdentifier=workload_config_name,
    RoleArn=role,
)

benchmark_job_arn = response["AIBenchmarkJobArn"]
print(f"Benchmark job created: {benchmark_job_name}")
print(f"   ARN: {benchmark_job_arn}")
print(f"   Output: {s3_output_location}")
print(f"\nWaiting for job to complete (typically 3-8 minutes)...")


In [ ]:
# Monitor the benchmark job until completion
job_response = wait_for_benchmark_job(
    job_name=benchmark_job_name,
    region=region,
    timeout_minutes=30,
)

## 4. Analyze Benchmark Results

The benchmark job writes detailed results to S3 in JSON format. Let's download and analyze them.

**Key metrics to understand:**
| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **TTFT** (Time to First Token) | Latency before first output token | User-perceived responsiveness |
| **ITL** (Inter-Token Latency) | Time between consecutive tokens | Streaming smoothness |
| **Request Latency** | Total end-to-end request time | Overall throughput capacity |
| **Throughput** (tokens/sec) | Output generation rate | System capacity |

In [ ]:
# Download and parse results from S3
results = parse_benchmark_results(s3_output_location, region=region)

if results:
    print("Raw results downloaded successfully!")
    print(f"   Keys: {list(results.keys())[:10]}")
else:
    print("No results found yet. The job may still be writing output.")
    print(f"   Check S3: {s3_output_location}")

In [ ]:
# Extract and display key metrics
metrics = extract_metrics(results)
format_metrics_table(metrics, title="Lab 1 Baseline - Llama 3.1 8B on ml.g5.12xlarge")

In [ ]:
# Save baseline metrics for comparison in Lab 4
import os
os.makedirs("../results", exist_ok=True)
with open("../results/lab1_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nBaseline metrics saved to: ../results/lab1_baseline_metrics.json")
print(f"These will be used for comparison in Lab 4.")

## 5. Interpreting Your Results

### What do these metrics tell us?

**Time to First Token (TTFT):**
- This is the "thinking time" before the model starts responding
- Includes: request queuing, prompt processing (prefill), and first token generation
- Lower is better - users perceive <500ms as "instant"
- At concurrency=10, you'll see higher TTFT than at concurrency=1 (queuing effect)

**Inter-Token Latency (ITL):**
- The gap between each streamed token
- Primarily driven by: model size, GPU memory bandwidth, and batch size
- Lower is better - <50ms feels smooth for streaming chat
- Relatively stable across concurrency levels (unlike TTFT)

**Throughput (tokens/second):**
- The system's total output generation capacity
- Higher is better - but there's always a latency tradeoff
- Increases with concurrency up to a saturation point

**Latency Percentiles (P50/P90/P99):**
- P50 = median request latency (typical user experience)
- P90 = 90th percentile (worst case for most users)
- P99 = 99th percentile (tail latency - important for SLAs)

> 💡 **Key insight**: There is always a tradeoff between throughput and latency. Increasing concurrency improves throughput but increases latency. In Lab 2, we'll explore this tradeoff systematically.

## 6. Cleanup

Delete this lab's Inference Component and workload config. The shared endpoint stays running.


In [ ]:
# Cleanup — delete IC, workload config, and secret
# (The shared endpoint stays running for other labs)
import time

# 1. Delete Inference Component
sm_client.delete_inference_component(InferenceComponentName=inference_component_name)
print(f"⏳ Deleting IC '{inference_component_name}'...")

while True:
    try:
        response = sm_client.describe_inference_component(
            InferenceComponentName=inference_component_name
        )
        print(f"  IC Status: {response['InferenceComponentStatus']}")
        time.sleep(15)
    except Exception as e:
        if "Could not find" in str(e) or "ValidationException" in str(e):
            print(f"  ✅ IC deleted")
            break
        raise

# 2. Delete Workload Config
sm_client.delete_ai_workload_config(AIWorkloadConfigName=workload_config_name)
print(f"✅ Workload config '{workload_config_name}' deleted")

# 3. Delete HF secret
print(f"✅ HF secret deleted")

print(f"\n✅ Lab 1 cleanup complete!")
print(f"   Shared endpoint '{endpoint_name}' is still running for other labs.")
print(f"   Run Lab 0 cleanup when you're done with ALL labs.")


---

## Summary

In this lab, you:
- Deployed Meta Llama 3.1 8B Instruct on SageMaker AI
- Created a workload configuration describing a typical chatbot workload
- Ran an automated benchmark job using NVIDIA AIPerf
- Analyzed TTFT, ITL, throughput, and latency percentiles
- Saved baseline metrics for comparison in Lab 4

**Next**: In [Lab 2](../lab2/lab2_benchmarking_nuances.ipynb), you'll explore how different workload parameters impact these metrics - varying concurrency, token lengths, and request rates to understand the performance characteristics of your deployment.